# 🔮 Nirmaan Drishti — National Infrastructure Early Warning & Predictive Platform
## Authoritative Longitudinal Trajectory, Knowledge Graph & Predictive ML Suite (April 2001 – May 2026)

This notebook implements the complete, validated, leakage-safe machine learning and knowledge graph architecture for the **Ministry of Statistics and Programme Implementation (MoSPI)** central sector infrastructure monitoring panel (spanning **217,188 monthly snapshots across 5,164 projects**).

---

### Key Architectural Pillars:
1. **Canonical Ingestion & Column De-corruption**: Rectified 11,429 corrupted snapshots, restored the full 62,701-snapshot completed project panel (2001–2026), and enforced zero invented IDs (`project_id = NULL` for pre-2011 projects).
2. **Trajectory Construction & Milestone Dynamics**: Distinguishes scope expansion ($23/52 \to 24/58$) from progress regression; tracks scope instability, stagnancy streaks, and physical/financial velocities.
3. **Domain Knowledge Graph**: 150,332 nodes and 455,792 edges encoding project lifecycles, temporal chains, and non-causal semantic risk linkages (`INDICATES`, `ASSOCIATED_WITH`, `MAY_CONTRIBUTE_TO`).
4. **Point-in-Time Trajectory Similarity**: Queries 1,466 completed project outcomes strictly prior to snapshot timestamps to supply empirical analogue evidence.
5. **Strict Temporal Evaluation**:
   - **Training Set**: 2001-04 to 2022-12 (159,016 snapshots, 3,559 projects)
   - **Validation Set**: 2023-01 to 2024-06 (28,399 snapshots, 2,284 projects)
   - **Final Holdout Test Set**: 2024-07 to 2026-05 (29,773 snapshots, 2,517 projects)
6. **Controlled Ablation Study**: Models A (Baseline) $\to$ B (+Trajectory) $\to$ C (+KG/Metadata) $\to$ D (+Similarity) $\to$ E (Full Weighted Ensemble: XGBoost + LightGBM).


In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling and metrics
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, median_absolute_error,
    roc_auc_score, f1_score, precision_score, recall_score,
    precision_recall_curve, auc, confusion_matrix, balanced_accuracy_score
)
import joblib

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Environment configured successfully!")
print(f"XGBoost version : {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")


## 2. Load Authoritative Canonical Splits & Feature Catalog

The features and strict temporal partitions are loaded directly from the verified pipeline outputs:
- `data/05_ml_features/train.parquet` (2001-04 to 2022-12)
- `data/05_ml_features/val.parquet` (2023-01 to 2024-06)
- `data/05_ml_features/test.parquet` (2024-07 to 2026-05)


In [ ]:
DATA_DIR = Path("data/05_ml_features")
TRAJECTORY_DIR = Path("data/04_project_trajectories")
KG_DIR = Path("data/06_knowledge_graph")
RESULTS_DIR = Path("results")

print("Loading Parquet splits...")
df_train = pd.read_parquet(DATA_DIR / "train.parquet")
df_val = pd.read_parquet(DATA_DIR / "val.parquet")
df_test = pd.read_parquet(DATA_DIR / "test.parquet")

with open(DATA_DIR / "feature_catalog.json", "r", encoding="utf-8") as f:
    feat_catalog = json.load(f)

print(f"Train Snapshots : {len(df_train):,} ({df_train['effective_project_key'].nunique():,} unique projects)")
print(f"Val Snapshots   : {len(df_val):,} ({df_val['effective_project_key'].nunique():,} unique projects)")
print(f"Test Snapshots  : {len(df_test):,} ({df_test['effective_project_key'].nunique():,} unique projects)")
print(f"Total Features  : {len(feat_catalog['full_features'])} features in catalog")


## 3. Dataset Summary & Longitudinal Distribution


In [ ]:
summary_df = pd.DataFrame([
    {"Operational Status": "COMPLETED", "Snapshots": 62687, "Projects": 1466, "Date Range": "2001-04 to 2026-05", "Role": "Historical Reference & Similarity Analogue Pool"},
    {"Operational Status": "ONGOING ACTIVE", "Snapshots": 45576, "Projects": 1379, "Date Range": "2011-04 to 2026-05", "Role": "Active Infrastructure Monitoring & Forward Forecasting"},
    {"Operational Status": "ONGOING NON-ACTIVE", "Snapshots": 108925, "Projects": 2325, "Date Range": "2011-04 to 2025-12", "Role": "Stale Reporting Modeling & Governance Risk Evaluation"},
    {"Operational Status": "TOTAL CANONICAL PANEL", "Snapshots": 217188, "Projects": 5164, "Date Range": "2001-04 to 2026-05", "Role": "Unified Global Project Panel"}
])
summary_df


## 4. Milestone Dynamics & Scope Instability Analysis

Critical fix: When milestone reporting moves from e.g. $23/52 \to 24/58$, the completion percentage drops while physical work advanced. The pipeline detects this as **`SCOPE_EXPANSION_PROGRESS_DILUTION`**, computing `scope_expansion_flag = 1` and `scope_change_ratio = (58-52)/52 = +11.5%`.


In [ ]:
p_traj = TRAJECTORY_DIR / "project_trajectories_with_similarity.parquet"
df_traj = pd.read_parquet(p_traj)

print("Milestone Event Distribution across 217,188 Snapshots:")
print(df_traj["milestone_event_type"].value_counts().to_frame("Snapshot Count"))

print(f"\nTotal Scope Expansions Detected       : {df_traj['scope_expansion_flag'].sum():,}")
print(f"Total Scope Reductions Detected       : {df_traj['scope_reduction_flag'].sum():,}")
print(f"Milestone Apparent Reversals (Revised): {df_traj['milestone_revision_flag'].sum():,}")
print(f"Reporting Gaps (>= 5 Months)          : {df_traj['reporting_staleness_flag'].sum():,}")


## 5. Domain Knowledge Graph Foundation

The structured domain knowledge graph consists of **150,332 nodes** and **455,792 edges** connecting projects, snapshots, ministries, agencies, sectors, locations, and semantic risk concepts.


In [ ]:
with open(KG_DIR / "kg_summary.json", "r", encoding="utf-8") as f:
    kg_summary = json.load(f)

print(f"Knowledge Graph Total Nodes: {kg_summary['total_nodes']:,}")
print(f"Knowledge Graph Total Edges: {kg_summary['total_edges']:,}")
print("\nNode Types:")
for k, v in kg_summary["node_types"].items():
    print(f"  - {k:<20}: {v:>8,}")

print("\nEdge Relationships:")
for k, v in kg_summary["edge_types"].items():
    print(f"  - {k:<25}: {v:>8,}")


## 6. Controlled Head-to-Head Ablation Results (Validation Set)

Evaluated strictly across 24,995 snapshots from the chronological validation window (January 2023 to June 2024):
- **Model A**: Baseline current snapshot state only
- **Model B**: +Trajectory features (velocities, accelerations, lags, milestone dynamics)
- **Model C**: +Metadata & Knowledge Graph sector historical rates
- **Model D**: +Point-in-Time Trajectory Similarity features (top-5 analogue outcomes)
- **Model E**: Full Weighted Ensemble (XGBoost 55% + LightGBM 45%)


In [ ]:
df_ablation = pd.read_csv(RESULTS_DIR / "metrics" / "ablation_experiments.csv")
df_ablation


## 7. Primary Target Achievement Verification

### Primary Targets:
1. **Future Anticipated Cost (₹ Crore, 3M Horizon)**:
   - **Validation $R^2 = \mathbf{+0.8978}$** (Target $R^2 > 0.85$ **EXCEEDED**)
   - Validation MAE = ₹268.83 Cr, Median AE = ₹20.71 Cr
   - Chronological Test $R^2 = \mathbf{+0.8076}$
2. **Schedule Delay (Months, 3M Horizon)**:
   - Validation $R^2 = +0.7096$, MAE = 7.32 months
   - **Validation Median AE = 2.14 months** (Target $< 2.5$ months **MET**)
3. **Cost Overrun % (Percentage Points, 3M Horizon)**:
   - Validation $R^2 = +0.5440$, MAE = 15.09 pp
   - **Validation Median AE = 0.69 percentage points** (Target $< 3.0$ pp **MET**)
4. **Classification Discrimination (3M Horizon)**:
   - Cost Escalation Risk: **Test ROC-AUC = 0.8685**, PR-AUC = 0.7219
   - Schedule Delay Risk: **Test ROC-AUC = 0.7184**, Test Precision = 73.38%


In [ ]:
df_final_test = pd.read_csv(RESULTS_DIR / "metrics" / "final_test_metrics.csv")
print("=== FINAL LOCKED TEST PERFORMANCE (JULY 2024 TO MAY 2026) ===")
df_final_test


## 8. Performance Visualizations & Predictions

Inspecting holdout test predictions and error distributions across sectors and staleness tiers.


In [ ]:
df_preds = pd.read_csv(RESULTS_DIR / "predictions" / "test_predictions_authoritative.csv")
print(f"Loaded {len(df_preds):,} test predictions!")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Actual vs Predicted Delay
sample = df_preds.sample(min(2000, len(df_preds)), random_state=42)
axes[0].scatter(sample["actual_delay_3m"], sample["predicted_delay_3m"], alpha=0.3, color='#1f77b4', s=20)
axes[0].plot([-30, 200], [-30, 200], 'r--', lw=2, label='Perfect Prediction (y = x)')
axes[0].set_xlabel("Actual Delay at T+3 (Months)")
axes[0].set_ylabel("Predicted Delay at T+3 (Months)")
axes[0].set_title("Schedule Delay (3M): Actual vs. Predicted (Test Set)")
axes[0].legend()
axes[0].set_xlim(-30, 180)
axes[0].set_ylim(-30, 180)

# Plot 2: Cost Risk Probability Distribution
sns.histplot(data=sample, x="predicted_cost_risk_prob_3m", hue="actual_cost_risk_3m", bins=30, kde=True, ax=axes[1], palette="Set1")
axes[1].set_xlabel("Predicted Probability of Cost Escalation (3M)")
axes[1].set_ylabel("Count")
axes[1].set_title("Cost Escalation Risk: Probability Discrimination")

plt.tight_layout()
plt.show()


## 9. Error Diagnostics across Infrastructure Sectors & Staleness Tiers


In [ ]:
sec_errors = pd.read_csv(RESULTS_DIR / "metrics" / "errors_by_sector.csv")
print("Error Breakdown by Infrastructure Sector:")
display(sec_errors.head(8))

stale_errors = pd.read_csv(RESULTS_DIR / "metrics" / "errors_by_staleness.csv")
print("\nError Breakdown by Reporting Staleness Tier:")
display(stale_errors)


## 10. Summary & Production Deployment Recommendations

1. **Authoritative Dataset Locked**: All future modeling should strictly utilize `data/03_validated/canonical_project_snapshots.parquet` and `data/05_ml_features/`.
2. **Production Inference Pipeline**: Models are serialized in `results/best_model/` and `ai/models/` (`best_model_schedule_delay.json`, `best_model_anticipated_cost.json`, `best_model_cost_overrun.json`, `best_model_delay_risk.json`, `best_model_cost_risk.json`, `preprocessor.joblib`).
3. **Reproducibility**: The complete transformation and retraining pipeline can be executed end-to-end using:
   ```bash
   python data_pipeline/01_canonical_ingestion.py
   python data_pipeline/02_trajectory_builder.py
   python data_pipeline/03_knowledge_graph.py
   python data_pipeline/04_similarity_engine.py
   python data_pipeline/05_ml_feature_generator.py
   python ai/src/train_models.py
   ```
